# 03: Model Interpretability — SHAP & Beyond

## Why Interpretability Matters

Machine learning models are increasingly used to make high-stakes decisions — loan approvals, medical diagnoses, hiring, criminal justice. But many powerful models (gradient boosting, neural networks) are **black boxes**: they give predictions without explaining *why*.

This creates real problems:

- **Trust**: Stakeholders won't adopt a model they don't understand
- **Debugging**: Without explanations, you can't tell if a model learned real patterns or spurious correlations
- **Compliance**: Regulations like GDPR and ECOA require explanations for automated decisions
- **Business adoption**: Decision-makers need to understand model logic to act on predictions

**Interpretability bridges the gap between model accuracy and real-world usefulness.**

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Distinguish** between global and local interpretability
2. **Use SHAP** (SHapley Additive exPlanations) for both global and local explanations
3. **Compare** multiple feature importance methods (built-in, permutation, SHAP)
4. **Create and interpret** partial dependence plots (PDPs)
5. **Generate waterfall and force plots** to explain individual predictions
6. **Choose** the right explainer for different model types

In [ ]:
# ============================================================
# Setup
# ============================================================
import sys
sys.path.insert(0, '../../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

from shared.viz_utils import setup_style, save_fig
setup_style()

# SHAP — the star of this notebook
try:
    import shap
    shap.initjs()
    print(f"SHAP version: {shap.__version__}")
except ImportError:
    print("Install SHAP: pip install shap")

print("Setup complete!")

---

# Section 1: Why Interpretability?

## The Black Box Problem

Many machine learning models work like this:

```
Input features  -->  [ ??? ]  -->  Prediction
```

We know the inputs and outputs, but the internal logic is opaque. This is especially true for:
- **Ensemble methods** (Random Forest, Gradient Boosting)
- **Deep neural networks**
- **Complex pipelines** with many transformations

## Two Levels of Interpretability

| Level | Question Answered | Methods | Use Case |
|-------|-------------------|---------|----------|
| **Global** | Which features matter most *overall*? | Feature importance, SHAP summary plot, Partial Dependence Plots | Model validation, feature selection, stakeholder communication |
| **Local** | Why did the model make *this specific* prediction? | SHAP waterfall plot, SHAP force plot, LIME | Individual decision explanations, debugging outliers, compliance |

### Who needs interpretability?

- **Regulators**: "Prove your loan model doesn't discriminate"
- **Business leaders**: "Why should I trust this model's recommendation?"
- **Data scientists**: "Is this model learning real patterns or noise?"
- **End users**: "Why was my application denied?"

> **Key insight**: The best model is not always the most accurate one — it's the one that is accurate AND understandable enough to be trusted and deployed.

---

## Generate a Synthetic Credit Risk Dataset

We'll create a realistic credit risk scenario with named features that are easy to interpret. The target variable is whether a customer **defaults** on a loan (1 = default, 0 = no default).

In [ ]:
# ============================================================
# Generate synthetic credit risk dataset
# ============================================================
np.random.seed(42)

# Create classification data with 8 informative features
X_raw, y = make_classification(
    n_samples=1000,
    n_features=8,
    n_informative=6,
    n_redundant=1,
    n_clusters_per_class=2,
    random_state=42,
    flip_y=0.05  # 5% label noise for realism
)

# Give features meaningful names (credit risk context)
feature_names = [
    'income', 'age', 'debt_ratio', 'credit_score',
    'employment_years', 'num_accounts', 'loan_amount', 'payment_history'
]

# Convert to DataFrame for readability
X_df = pd.DataFrame(X_raw, columns=feature_names)

# Scale features to realistic ranges for interpretability
# (This helps us reason about the features more naturally)
X_df['income'] = X_df['income'] * 15000 + 55000           # ~$25k-$85k
X_df['age'] = X_df['age'] * 8 + 40                         # ~24-56
X_df['debt_ratio'] = np.clip(X_df['debt_ratio'] * 0.15 + 0.35, 0, 1)  # 0-100%
X_df['credit_score'] = X_df['credit_score'] * 80 + 680     # ~520-840
X_df['employment_years'] = np.clip(X_df['employment_years'] * 4 + 8, 0, 30)  # 0-30
X_df['num_accounts'] = np.clip(np.round(X_df['num_accounts'] * 3 + 5), 0, 15)  # 0-15
X_df['loan_amount'] = X_df['loan_amount'] * 8000 + 20000   # ~$4k-$36k
X_df['payment_history'] = np.clip(X_df['payment_history'] * 0.15 + 0.8, 0, 1)  # 0-1

print("Dataset shape:", X_df.shape)
print(f"Default rate: {y.mean():.1%}")
print(f"\nFeature summary:")
print(X_df.describe().round(2))
print(f"\nTarget distribution: 0 (no default) = {(y==0).sum()}, 1 (default) = {(y==1).sum()}")

In [ ]:
# ============================================================
# Train-test split and model training
# ============================================================

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, random_state=42, stratify=y
)

# Keep DataFrames for SHAP (feature names matter!)
X_train_df = X_train.copy()
X_test_df = X_test.copy()

# Train a Gradient Boosting Classifier — a powerful "black box" model
model = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    random_state=42
)
model.fit(X_train_df, y_train)

# Evaluate
train_acc = accuracy_score(y_train, model.predict(X_train_df))
test_acc = accuracy_score(y_test, model.predict(X_test_df))

print(f"Model: GradientBoostingClassifier")
print(f"Train accuracy: {train_acc:.3f}")
print(f"Test accuracy:  {test_acc:.3f}")
print(f"\nNow let's figure out WHY it makes these predictions...")

---

# Section 2: Built-in Feature Importance

## sklearn's `feature_importances_`

Tree-based models in sklearn provide a built-in `feature_importances_` attribute. This measures the **mean decrease in impurity** (MDI) — how much each feature reduces the Gini impurity (or entropy) across all trees.

### How it works:
1. For each tree, track how much each feature reduces impurity when used in a split
2. Average this across all trees
3. Normalize so all importances sum to 1.0

### Caveats:
- **Biased toward high-cardinality features**: Features with many unique values (like continuous features) tend to get higher importance, even if they're not truly more important
- **Affected by correlated features**: If two features are correlated, the importance gets split between them, making both look less important than they are
- **Doesn't show direction**: You know a feature is important, but not whether high values increase or decrease the prediction

Despite these limitations, built-in importance is a great **quick first look**.

In [ ]:
# ============================================================
# Built-in Feature Importance (Mean Decrease in Impurity)
# ============================================================

# Get built-in importances
importances = model.feature_importances_
sorted_idx = np.argsort(importances)

fig, ax = plt.subplots(figsize=(8, 5))

# Horizontal bar chart sorted by importance
ax.barh(
    range(len(sorted_idx)),
    importances[sorted_idx],
    color=plt.cm.viridis(np.linspace(0.3, 0.9, len(sorted_idx)))
)
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels(np.array(feature_names)[sorted_idx])
ax.set_xlabel('Feature Importance (Mean Decrease in Impurity)')
ax.set_title('Built-in Feature Importance — GradientBoosting')

# Add value labels on bars
for i, (val, name) in enumerate(zip(importances[sorted_idx], np.array(feature_names)[sorted_idx])):
    ax.text(val + 0.005, i, f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
save_fig(fig, 'builtin_feature_importance')
plt.show()

print("Top 3 features (built-in importance):")
for idx in sorted_idx[::-1][:3]:
    print(f"  {feature_names[idx]}: {importances[idx]:.3f}")

---

# Section 3: Permutation Importance

## A More Reliable Alternative

**Permutation importance** measures how much the model's performance *drops* when a single feature is randomly shuffled. The idea is simple:

1. Compute the model's baseline accuracy on the test set
2. For each feature:
   - Randomly shuffle that feature's values (breaking its relationship with the target)
   - Re-compute accuracy
   - The **drop in accuracy** = that feature's importance
3. Repeat multiple times for stable estimates

### Advantages over built-in importance:
- **Works with any model** (not just tree-based)
- **Less biased** toward high-cardinality features
- **Computed on test data**, so it reflects real predictive power
- **Provides uncertainty estimates** (from multiple shuffles)

### Limitation:
- Can underestimate importance of **correlated features** (shuffling one still leaves the correlated partner intact)

In [ ]:
# ============================================================
# Permutation Importance
# ============================================================

# Compute permutation importance on the TEST set
perm_importance = permutation_importance(
    model, X_test_df, y_test,
    n_repeats=30,       # Repeat 30 times for stable estimates
    random_state=42,
    scoring='accuracy'
)

# Sort by mean importance
perm_sorted_idx = perm_importance.importances_mean.argsort()

# ---- Side-by-side comparison ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Built-in importance
axes[0].barh(
    range(len(sorted_idx)),
    importances[sorted_idx],
    color='steelblue', alpha=0.8
)
axes[0].set_yticks(range(len(sorted_idx)))
axes[0].set_yticklabels(np.array(feature_names)[sorted_idx])
axes[0].set_xlabel('Importance')
axes[0].set_title('Built-in (Mean Decrease in Impurity)')

# Right: Permutation importance with error bars
axes[1].barh(
    range(len(perm_sorted_idx)),
    perm_importance.importances_mean[perm_sorted_idx],
    xerr=perm_importance.importances_std[perm_sorted_idx],
    color='coral', alpha=0.8, capsize=3
)
axes[1].set_yticks(range(len(perm_sorted_idx)))
axes[1].set_yticklabels(np.array(feature_names)[perm_sorted_idx])
axes[1].set_xlabel('Mean Accuracy Decrease')
axes[1].set_title('Permutation Importance (on Test Set)')

plt.suptitle('Comparing Feature Importance Methods', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, 'importance_comparison')
plt.show()

print("Note: Error bars on permutation importance show variability across shuffles.")
print("Features with large error bars have less stable importance estimates.")

---

# Section 4: Partial Dependence Plots (PDPs)

## Understanding Feature *Effects* (Not Just Importance)

Feature importance tells you **which** features matter, but not **how** they affect predictions. Partial Dependence Plots fill this gap.

### How PDPs work:
1. Pick a feature (e.g., `credit_score`)
2. For each value of that feature (e.g., 550, 600, 650, 700, ...):
   - Set ALL samples to that value for the chosen feature
   - Keep all other features unchanged
   - Average the model's predictions
3. Plot the average prediction vs. the feature value

### What you learn:
- **Monotonic relationships**: "Higher credit score = lower default probability" (makes sense!)
- **Non-linear effects**: "Income matters a lot below $40k but plateaus above $60k"
- **Thresholds**: "Default risk jumps sharply when debt_ratio exceeds 0.5"

### 2D PDPs:
You can also plot two features together to see **interactions** — how the combined effect of two features differs from their individual effects.

In [ ]:
# ============================================================
# Partial Dependence Plots — Top 3 Features
# ============================================================

# Identify top 3 features by built-in importance
top3_idx = np.argsort(importances)[::-1][:3]
top3_names = [feature_names[i] for i in top3_idx]
print(f"Plotting PDPs for top 3 features: {top3_names}\n")

# 1D Partial Dependence Plots
fig, ax = plt.subplots(figsize=(14, 4))
display = PartialDependenceDisplay.from_estimator(
    model,
    X_train_df,
    features=top3_idx.tolist(),
    feature_names=feature_names,
    kind='both',    # Show both average PDP line AND individual ICE curves
    ax=ax,
    ice_lines_kw={'alpha': 0.05, 'color': 'steelblue'},
    pd_line_kw={'color': 'red', 'linewidth': 2}
)
fig.suptitle('Partial Dependence Plots (with ICE curves in light blue)', 
             fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
save_fig(fig, 'pdp_top3')
plt.show()

print("Red line = average partial dependence (PDP)")
print("Light blue lines = Individual Conditional Expectation (ICE) curves")
print("  -> ICE shows how the prediction changes for each individual sample")

In [ ]:
# ============================================================
# 2D Partial Dependence Plot — Feature Interaction
# ============================================================

# Show how the top 2 features interact
top2_idx = tuple(np.argsort(importances)[::-1][:2].tolist())
print(f"2D PDP: interaction between {feature_names[top2_idx[0]]} and {feature_names[top2_idx[1]]}\n")

fig, ax = plt.subplots(figsize=(8, 6))
display_2d = PartialDependenceDisplay.from_estimator(
    model,
    X_train_df,
    features=[top2_idx],  # Tuple inside list = 2D interaction
    feature_names=feature_names,
    ax=ax,
    kind='average'
)
fig.suptitle(f'2D Partial Dependence: {feature_names[top2_idx[0]]} vs {feature_names[top2_idx[1]]}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'pdp_2d_interaction')
plt.show()

print("The contour plot shows how the predicted probability changes")
print("as both features vary simultaneously. Look for non-additive patterns!")

---

# Section 5: SHAP — The Gold Standard of Interpretability

## What is SHAP?

**SHAP** (SHapley Additive exPlanations) is based on **Shapley values** from cooperative game theory. The core idea:

> Imagine each feature is a "player" in a game, and the "payout" is the prediction. SHAP assigns each feature a fair share of the credit for the final prediction.

### How Shapley values work (intuitively):
1. Consider all possible subsets of features
2. For each subset, measure how much adding feature X changes the prediction
3. Average this contribution across all possible subsets
4. This average = the Shapley value for feature X

### Three key properties:
- **Local accuracy**: The SHAP values for a prediction sum up to the difference between the prediction and the average prediction (base value)
- **Missingness**: If a feature is missing (not used), its SHAP value is 0
- **Consistency**: If a feature's contribution increases in a new model, its SHAP value never decreases

### Why SHAP is special:
- Provides **both global AND local** explanations from the same computation
- Has a **solid theoretical foundation** (game theory)
- **Direction-aware**: positive SHAP = pushes prediction up, negative SHAP = pushes prediction down
- **Additive**: `base_value + sum(SHAP values) = model prediction`

### SHAP value interpretation:
- **Positive SHAP value** for a feature means it pushed the prediction **higher** (toward class 1 / default)
- **Negative SHAP value** means it pushed the prediction **lower** (toward class 0 / no default)
- The **magnitude** tells you how strongly the feature influenced this particular prediction

In [ ]:
# ============================================================
# SHAP TreeExplainer — Computing SHAP Values
# ============================================================

# TreeExplainer is optimized for tree-based models (fast & exact)
explainer = shap.TreeExplainer(model)

# Compute SHAP values for the test set
# For binary classification, we get SHAP values for class 1 (default)
shap_values = explainer.shap_values(X_test_df)

# Understand the output
print("=== Understanding SHAP Output ===")
print(f"X_test shape:    {X_test_df.shape}")
print(f"SHAP values shape: {np.array(shap_values).shape}")
print(f"  -> One SHAP value per sample per feature")
print(f"\nBase value (expected value): {explainer.expected_value:.4f}")
print(f"  -> This is the average model output (log-odds for GBM)")
print(f"\n--- Verification: SHAP values are additive ---")
sample_idx = 0
shap_sum = np.sum(shap_values[sample_idx])
prediction = model.decision_function(X_test_df.iloc[[sample_idx]])[0]
print(f"Sample {sample_idx}:")
print(f"  Base value + sum(SHAP) = {explainer.expected_value:.4f} + {shap_sum:.4f} = {explainer.expected_value + shap_sum:.4f}")
print(f"  Model decision function = {prediction:.4f}")
print(f"  Match: {np.isclose(explainer.expected_value + shap_sum, prediction)}")

---

# Section 6: SHAP Summary Plot

## "The Single Most Informative Plot in ML Interpretability"

The SHAP **beeswarm summary plot** packs an incredible amount of information into one visualization:

- **Each dot** = one sample (one customer in our case)
- **Y-axis** = features, sorted by overall importance (top = most important)
- **X-axis** = SHAP value (how much this feature pushed the prediction for this sample)
- **Color** = the actual feature value (red = high, blue = low)

### How to read it:
- Dots to the **right** (positive SHAP) pushed the prediction toward default
- Dots to the **left** (negative SHAP) pushed the prediction away from default
- If red dots cluster on the right, it means **high values of that feature increase default risk**
- If blue dots cluster on the right, it means **low values increase default risk**
- Wide spread = the feature has a large effect; narrow = small effect

In [ ]:
# ============================================================
# SHAP Summary Plot — Beeswarm
# ============================================================

# The beeswarm plot: the most informative single plot in ML
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_df, show=False)
plt.title('SHAP Summary Plot (Beeswarm)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("How to read this plot:")
print("  - Each dot is one customer prediction")
print("  - X-axis: SHAP value (positive = pushes toward default)")
print("  - Color: red = high feature value, blue = low feature value")
print("  - Features are sorted top-to-bottom by importance")

In [ ]:
# ============================================================
# SHAP Summary Plot — Bar Version (simpler, for presentations)
# ============================================================

# The bar version shows mean absolute SHAP value per feature
# Great for stakeholder presentations where beeswarm is too complex
plt.figure(figsize=(10, 5))
shap.summary_plot(shap_values, X_test_df, plot_type="bar", show=False)
plt.title('SHAP Feature Importance (Mean |SHAP Value|)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("The bar plot shows the average magnitude of SHAP values per feature.")
print("This is a global importance measure — similar to permutation importance,")
print("but derived from SHAP's theoretically grounded framework.")

---

# Section 7: SHAP Dependence Plots

## Like PDPs, But Better

SHAP dependence plots are similar to Partial Dependence Plots but with key advantages:

- **Show individual data points** (not just averages), so you can see the full distribution
- **Automatically detect interactions**: the plot colors dots by the feature that interacts most strongly with the chosen feature
- **Based on SHAP values**, which account for all feature interactions

### Reading the plot:
- **X-axis**: actual feature value
- **Y-axis**: SHAP value for that feature (how much it contributed to each prediction)
- **Color**: value of the most interacting feature (automatically chosen by SHAP)
- A **vertical spread** at a given x-value means other features are modifying this feature's effect (interaction!)

In [ ]:
# ============================================================
# SHAP Dependence Plots — Top 3 Features
# ============================================================

# Plot SHAP dependence for the top 3 most important features
top3_shap = np.argsort(np.abs(shap_values).mean(axis=0))[::-1][:3]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, feat_idx in enumerate(top3_shap):
    plt.sca(axes[i])
    shap.dependence_plot(
        feat_idx,
        shap_values,
        X_test_df,
        ax=axes[i],
        show=False
    )
    axes[i].set_title(f'SHAP Dependence: {feature_names[feat_idx]}', fontsize=11)

plt.suptitle('SHAP Dependence Plots — Top 3 Features', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
save_fig(fig, 'shap_dependence_top3')
plt.show()

print("The color in each plot represents the feature SHAP automatically detected")
print("as having the strongest interaction with the plotted feature.")
print("\nVertical spread at a given x-value = the effect of that feature depends")
print("on other features (interaction effect).")

---

# Section 8: Local Explanations — Why THIS Prediction?

## From Global to Local

So far we've looked at global patterns (which features matter overall). But often we need to explain **individual predictions**:

- "Why was this customer's loan denied?"
- "What factors contributed most to this patient's risk score?"
- "Why did the model flag this transaction as fraud?"

SHAP provides two powerful local explanation tools:

### Waterfall Plot
- Shows the **step-by-step construction** of a single prediction
- Starts from the base value (average prediction) and adds/subtracts each feature's contribution
- Easy to read: "Starting from the average, feature X added +0.3, feature Y subtracted -0.15, ..."

### Force Plot
- A more compact visualization of the same information
- Red features push the prediction higher, blue features push it lower
- The width of each colored segment shows the magnitude of the effect

In [ ]:
# ============================================================
# Local Explanations — Pick specific predictions to explain
# ============================================================

# Find one predicted-default and one predicted-no-default
predictions = model.predict(X_test_df)
pred_proba = model.predict_proba(X_test_df)[:, 1]

# Find a confident "default" prediction and a confident "no default" prediction
default_idx = np.where(predictions == 1)[0]
no_default_idx = np.where(predictions == 0)[0]

# Pick the most confident examples
high_risk_idx = default_idx[np.argmax(pred_proba[default_idx])]
low_risk_idx = no_default_idx[np.argmin(pred_proba[no_default_idx])]

print("=== Two Customers to Explain ===\n")

print(f"Customer A (HIGH RISK) — index {high_risk_idx}")
print(f"  Predicted probability of default: {pred_proba[high_risk_idx]:.3f}")
print(f"  Actual label: {'Default' if y_test[high_risk_idx] == 1 else 'No default'}")
print(f"  Features:")
for feat, val in X_test_df.iloc[high_risk_idx].items():
    print(f"    {feat}: {val:.2f}")

print(f"\nCustomer B (LOW RISK) — index {low_risk_idx}")
print(f"  Predicted probability of default: {pred_proba[low_risk_idx]:.3f}")
print(f"  Actual label: {'Default' if y_test[low_risk_idx] == 1 else 'No default'}")
print(f"  Features:")
for feat, val in X_test_df.iloc[low_risk_idx].items():
    print(f"    {feat}: {val:.2f}")

In [ ]:
# ============================================================
# Waterfall Plots — Step-by-step explanation of each prediction
# ============================================================

# Use the newer SHAP Explanation object for waterfall plots
shap_explanation = explainer(X_test_df)

# --- Customer A: HIGH RISK ---
print("Customer A (HIGH RISK) — Waterfall Plot")
print("Read bottom-to-top: starting from E[f(x)], each feature adds or subtracts.\n")

plt.figure(figsize=(10, 6))
shap.waterfall_plot(shap_explanation[high_risk_idx], show=False)
plt.title('Customer A (HIGH RISK) — Why did the model predict default?', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# --- Customer B: LOW RISK ---
print("Customer B (LOW RISK) — Waterfall Plot")
print("Notice how the features push in the OPPOSITE direction.\n")

plt.figure(figsize=(10, 6))
shap.waterfall_plot(shap_explanation[low_risk_idx], show=False)
plt.title('Customer B (LOW RISK) — Why did the model predict no default?', fontsize=12)
plt.tight_layout()
plt.show()

print("\nHow to read waterfall plots:")
print("  - E[f(x)] at the bottom is the base value (average prediction)")
print("  - Red bars push the prediction HIGHER (toward default)")
print("  - Blue bars push the prediction LOWER (away from default)")
print("  - f(x) at the top is the final prediction for this customer")

In [ ]:
# ============================================================
# Force Plots — Compact local explanations
# ============================================================

# Force plot for Customer A (HIGH RISK)
print("Customer A (HIGH RISK) — Force Plot")
print("Red features push prediction UP (toward default)")
print("Blue features push prediction DOWN (away from default)\n")

shap.force_plot(
    explainer.expected_value,
    shap_values[high_risk_idx],
    X_test_df.iloc[high_risk_idx],
    matplotlib=True,
    show=False
)
plt.title('Customer A (HIGH RISK)', fontsize=12, y=1.25)
plt.tight_layout()
plt.show()

In [ ]:
# Force plot for Customer B (LOW RISK)
print("Customer B (LOW RISK) — Force Plot\n")

shap.force_plot(
    explainer.expected_value,
    shap_values[low_risk_idx],
    X_test_df.iloc[low_risk_idx],
    matplotlib=True,
    show=False
)
plt.title('Customer B (LOW RISK)', fontsize=12, y=1.25)
plt.tight_layout()
plt.show()

print("\nForce plot summary:")
print("  - The bold number is the model's prediction (log-odds)")
print("  - The base value is where the model starts (average)")
print("  - Red segments: features pushing the prediction UP")
print("  - Blue segments: features pushing the prediction DOWN")
print("  - Wider segments = larger contributions")

---

# Section 9: SHAP for Different Model Types

SHAP provides specialized explainers optimized for different model architectures:

| Explainer | Best For | Speed | Exactness |
|-----------|----------|-------|-----------|
| **TreeExplainer** | Tree-based models (RF, GBM, XGBoost, LightGBM) | Very fast | Exact |
| **LinearExplainer** | Linear models (LogisticRegression, LinearSVM) | Very fast | Exact |
| **KernelExplainer** | Any model (model-agnostic) | Slow | Approximate |
| **DeepExplainer** | Deep learning (TensorFlow, PyTorch) | Medium | Approximate |

### Choosing the right explainer:
- **Always use the specialized explainer** when available (TreeExplainer for trees, LinearExplainer for linear models)
- **KernelExplainer** is a fallback for any model, but it's much slower because it needs to evaluate the model many times
- For large datasets, use a **background sample** (e.g., `shap.sample(X_train, 100)`) to speed up KernelExplainer

Let's compare SHAP for a linear model with its native coefficients.

In [ ]:
# ============================================================
# SHAP LinearExplainer — Compare with model coefficients
# ============================================================

# Scale features for logistic regression (it needs scaled inputs)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_df),
    columns=feature_names,
    index=X_train_df.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_df),
    columns=feature_names,
    index=X_test_df.index
)

# Train logistic regression
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
lr_acc = accuracy_score(y_test, lr_model.predict(X_test_scaled))
print(f"Logistic Regression accuracy: {lr_acc:.3f}")

# SHAP LinearExplainer
lr_explainer = shap.LinearExplainer(lr_model, X_train_scaled)
lr_shap_values = lr_explainer.shap_values(X_test_scaled)

# Compare: model coefficients vs SHAP importance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Model coefficients
coef_sorted_idx = np.argsort(np.abs(lr_model.coef_[0]))
axes[0].barh(
    range(len(coef_sorted_idx)),
    lr_model.coef_[0][coef_sorted_idx],
    color=['coral' if c > 0 else 'steelblue' for c in lr_model.coef_[0][coef_sorted_idx]]
)
axes[0].set_yticks(range(len(coef_sorted_idx)))
axes[0].set_yticklabels(np.array(feature_names)[coef_sorted_idx])
axes[0].set_xlabel('Coefficient Value')
axes[0].set_title('Logistic Regression Coefficients')
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)

# Right: SHAP importance
mean_abs_shap = np.abs(lr_shap_values).mean(axis=0)
shap_sorted_idx = np.argsort(mean_abs_shap)
axes[1].barh(
    range(len(shap_sorted_idx)),
    mean_abs_shap[shap_sorted_idx],
    color='mediumpurple', alpha=0.8
)
axes[1].set_yticks(range(len(shap_sorted_idx)))
axes[1].set_yticklabels(np.array(feature_names)[shap_sorted_idx])
axes[1].set_xlabel('Mean |SHAP Value|')
axes[1].set_title('SHAP Importance (LinearExplainer)')

plt.suptitle('Linear Model: Coefficients vs SHAP', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, 'linear_coef_vs_shap')
plt.show()

print("For linear models, SHAP values and coefficients tell a similar story.")
print("SHAP has the advantage of being on the same scale as the prediction,")
print("making it easier to compare across different model types.")

---

# Section 10: Practical Tips for Using SHAP

## Best Practices

### 1. Preprocessing and feature names
Always compute SHAP values on data with **meaningful feature names**. Using DataFrames (not raw numpy arrays) makes plots much more readable. If you scale your features, do it *before* computing SHAP so the plots make sense.

### 2. Choose the right explainer
Use `TreeExplainer` for tree models and `LinearExplainer` for linear models whenever possible. Only fall back to `KernelExplainer` when you must (it's 10-100x slower).

### 3. Background data for large datasets
`KernelExplainer` and some other explainers need a "background dataset" to compute expected values. For large datasets, use a subsample:
```python
background = shap.sample(X_train, 100)
explainer = shap.KernelExplainer(model.predict_proba, background)
```

### 4. SHAP values are additive
This is the most important property to remember:
```
base_value + sum(SHAP values for all features) = model prediction
```
You can always verify your SHAP values sum correctly.

### 5. Save SHAP values for dashboards
SHAP values are just numpy arrays — save them for later use in dashboards or reports:
```python
np.save('shap_values.npy', shap_values)
```

### 6. Correlation caveat
If features are highly correlated, SHAP may distribute credit between them in unexpected ways. Consider feature selection or grouping correlated features before interpretation.

### 7. SHAP is not causal
SHAP tells you what the model *learned*, not what truly *causes* the outcome. A feature with high SHAP importance might be a proxy for something else.

---

# Key Takeaways

## What We Learned

1. **Feature importance is not the same as feature effect**
   - Importance = magnitude (how much does this feature matter?)
   - SHAP = direction + magnitude (does a high value increase or decrease the prediction?)

2. **Always cross-check with multiple methods**
   - Built-in importance (quick but biased)
   - Permutation importance (reliable, works with any model)
   - SHAP (gold standard, theoretically grounded)

3. **Your go-to interpretability toolkit**
   - **Global understanding**: SHAP summary (beeswarm) plot
   - **Feature effects**: SHAP dependence plots or PDPs
   - **Individual predictions**: SHAP waterfall plot
   - **Presentations**: SHAP bar plot for feature importance

4. **Use interpretability to validate your model**
   - If the model says "age" is the most important feature for credit risk, does that make business sense?
   - If a feature that shouldn't matter (like customer ID) shows up as important, something is wrong
   - Interpretability is a **debugging tool**, not just a communication tool

5. **Match your explainer to your model**
   - TreeExplainer for tree-based models (fast, exact)
   - LinearExplainer for linear models
   - KernelExplainer as a universal fallback

> **Bottom line**: A model you can't explain is a model you can't trust. SHAP gives you the tools to bridge that gap.

---

# Exercises

## Exercise 1: SHAP on Breast Cancer Dataset

Train a `RandomForestClassifier` on sklearn's `breast_cancer` dataset. Generate:
1. A SHAP summary (beeswarm) plot
2. A built-in `feature_importances_` bar plot
3. Compare the top 5 features from each method — do they agree?

In [ ]:
# ============================================================
# Exercise 1: Your code here
# ============================================================
from sklearn.datasets import load_breast_cancer

# Step 1: Load data
data = load_breast_cancer()
X_bc = pd.DataFrame(data.data, columns=data.feature_names)
y_bc = data.target

# Step 2: Train-test split
# X_bc_train, X_bc_test, y_bc_train, y_bc_test = ...

# Step 3: Train RandomForestClassifier
# rf_model = ...

# Step 4: Compute SHAP values using TreeExplainer
# rf_explainer = ...
# rf_shap_values = ...

# Step 5: Plot SHAP summary (beeswarm)
# shap.summary_plot(...)

# Step 6: Plot built-in feature importances
# ...

# Step 7: Compare top 5 features from each method
# ...

## Exercise 2: Find the Most "Surprising" Prediction

Using the credit risk model from this notebook:
1. Find the test sample where a single feature has the **largest absolute SHAP value** (i.e., one feature dominated the prediction)
2. Generate a waterfall plot for that sample
3. Explain in a comment: what made this prediction surprising or unusual?

*Hint*: Use `np.abs(shap_values).max(axis=1)` to find the sample with the most extreme single-feature contribution.

In [ ]:
# ============================================================
# Exercise 2: Your code here
# ============================================================

# Step 1: Find the sample with the largest single-feature SHAP value
# max_shap_per_sample = np.abs(shap_values).max(axis=1)
# surprising_idx = np.argmax(max_shap_per_sample)

# Step 2: Print the sample's features and prediction
# print(f"Most 'surprising' sample index: {surprising_idx}")
# print(f"Features:\n{X_test_df.iloc[surprising_idx]}")
# print(f"Prediction: {predictions[surprising_idx]}")

# Step 3: Generate waterfall plot
# shap.waterfall_plot(shap_explanation[surprising_idx])

# Step 4: Explain what you see (add a comment)
# ...

## Exercise 3: Cross-Model SHAP Comparison

Do different models agree on which features are important?

1. Train a `RandomForestClassifier` on the same credit risk data (use `X_train_df`, `y_train`)
2. Compute SHAP values for both the RandomForest and the GradientBoosting model (already trained)
3. Create a scatter plot comparing the mean |SHAP value| for each feature across both models
4. Do they agree on the top 3 features? What about the bottom 3?

*This is a great way to validate that your interpretability results are robust and not model-specific.*

In [ ]:
# ============================================================
# Exercise 3: Your code here
# ============================================================

# Step 1: Train a RandomForestClassifier
# rf_credit = RandomForestClassifier(n_estimators=200, random_state=42)
# rf_credit.fit(X_train_df, y_train)

# Step 2: Compute SHAP values for RandomForest
# rf_explainer = shap.TreeExplainer(rf_credit)
# rf_shap_vals = rf_explainer.shap_values(X_test_df)
# If rf_shap_vals is a list (binary classification), use rf_shap_vals[1] for class 1

# Step 3: Compute mean |SHAP| for both models
# gbm_importance = np.abs(shap_values).mean(axis=0)  # Already computed above
# rf_importance = np.abs(rf_shap_vals[1]).mean(axis=0)

# Step 4: Scatter plot — does GBM agree with RF?
# fig, ax = plt.subplots(figsize=(8, 8))
# ax.scatter(gbm_importance, rf_importance, s=100)
# for i, name in enumerate(feature_names):
#     ax.annotate(name, (gbm_importance[i], rf_importance[i]), fontsize=10)
# ax.set_xlabel('GradientBoosting Mean |SHAP|')
# ax.set_ylabel('RandomForest Mean |SHAP|')
# ax.set_title('Do Different Models Agree on Feature Importance?')
# ax.plot([0, max(gbm_importance)], [0, max(gbm_importance)], 'k--', alpha=0.3)
# plt.tight_layout()
# plt.show()

# Step 5: Compare top 3 features from each
# ...

---

*Notebook created for Module 2, Week 6 of the Applied AI Engineer Upgrade course.*  
*Topics: Model interpretability, SHAP, feature importance, partial dependence plots, local explanations.*